In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Configuration
load_dotenv()
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Connexion à la base de données
DATABASE_URL = os.getenv('DATABASE_URL')
engine = create_engine(DATABASE_URL)

print("✅ Environnement configuré")

## 1. Chargement des Données

In [ ]:
# Charger les plats
query_dishes = """
SELECT 
    d.id,
    d.name,
    d.meal_type,
    d.cuisine_origin,
    d.description
FROM dishes d
"""

df_dishes = pd.read_sql(query_dishes, engine)
print(f"📊 Nombre de plats : {len(df_dishes)}")
df_dishes.head()

In [ ]:
# Charger les données nutritionnelles
query_nutrition = """
SELECT 
    f.id,
    f.local_name,
    f.category,
    f.glycemic_index,
    f.nutritional_values::json->>'energy_kcal' as energy_kcal,
    f.nutritional_values::json->>'protein_g' as protein_g,
    f.nutritional_values::json->>'fat_g' as fat_g,
    f.nutritional_values::json->>'carbohydrate_g' as carbohydrate_g,
    f.nutritional_values::json->>'fiber_g' as fiber_g,
    f.nutritional_values::json->>'sodium_mg' as sodium_mg,
    f.nutritional_values::json->>'potassium_mg' as potassium_mg
FROM foods f
"""

df_nutrition = pd.read_sql(query_nutrition, engine)

# Convertir en numérique
numeric_cols = ['energy_kcal', 'protein_g', 'fat_g', 'carbohydrate_g', 'fiber_g', 'sodium_mg', 'potassium_mg']
for col in numeric_cols:
    df_nutrition[col] = pd.to_numeric(df_nutrition[col], errors='coerce')

print(f"📊 Nombre d'aliments : {len(df_nutrition)}")
df_nutrition.head()

## 2. Statistiques Descriptives

In [ ]:
# Résumé statistique
print("=" * 60)
print("STATISTIQUES NUTRITIONNELLES (par 100g)")
print("=" * 60)
df_nutrition[numeric_cols].describe().round(2)

In [ ]:
# Distribution des types de repas
meal_counts = df_dishes['meal_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(meal_counts.values, labels=meal_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Distribution des Types de Repas', fontsize=14, fontweight='bold')

# Bar chart
meal_counts.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Nombre de Plats par Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Type de Repas')
axes[1].set_ylabel('Nombre de Plats')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Analyse des Macronutriments

In [ ]:
# Distribution des macronutriments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Calories
axes[0, 0].hist(df_nutrition['energy_kcal'].dropna(), bins=30, color='coral', edgecolor='black')
axes[0, 0].set_title('Distribution des Calories (kcal/100g)', fontweight='bold')
axes[0, 0].set_xlabel('Calories')
axes[0, 0].set_ylabel('Fréquence')
axes[0, 0].axvline(df_nutrition['energy_kcal'].mean(), color='red', linestyle='--', label=f"Moyenne: {df_nutrition['energy_kcal'].mean():.1f}")
axes[0, 0].legend()

# Protéines
axes[0, 1].hist(df_nutrition['protein_g'].dropna(), bins=30, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Distribution des Protéines (g/100g)', fontweight='bold')
axes[0, 1].set_xlabel('Protéines')
axes[0, 1].set_ylabel('Fréquence')
axes[0, 1].axvline(df_nutrition['protein_g'].mean(), color='green', linestyle='--', label=f"Moyenne: {df_nutrition['protein_g'].mean():.1f}")
axes[0, 1].legend()

# Glucides
axes[1, 0].hist(df_nutrition['carbohydrate_g'].dropna(), bins=30, color='lightblue', edgecolor='black')
axes[1, 0].set_title('Distribution des Glucides (g/100g)', fontweight='bold')
axes[1, 0].set_xlabel('Glucides')
axes[1, 0].set_ylabel('Fréquence')
axes[1, 0].axvline(df_nutrition['carbohydrate_g'].mean(), color='blue', linestyle='--', label=f"Moyenne: {df_nutrition['carbohydrate_g'].mean():.1f}")
axes[1, 0].legend()

# Lipides
axes[1, 1].hist(df_nutrition['fat_g'].dropna(), bins=30, color='gold', edgecolor='black')
axes[1, 1].set_title('Distribution des Lipides (g/100g)', fontweight='bold')
axes[1, 1].set_xlabel('Lipides')
axes[1, 1].set_ylabel('Fréquence')
axes[1, 1].axvline(df_nutrition['fat_g'].mean(), color='orange', linestyle='--', label=f"Moyenne: {df_nutrition['fat_g'].mean():.1f}")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Corrélations Nutritionnelles

In [ ]:
# Matrice de corrélation
correlation_matrix = df_nutrition[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.2f')
plt.title('Matrice de Corrélation - Valeurs Nutritionnelles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Analyse par Catégorie

In [ ]:
# Calories moyennes par catégorie
category_stats = df_nutrition.groupby('category')['energy_kcal'].agg(['mean', 'count']).sort_values('mean', ascending=False)
category_stats = category_stats[category_stats['count'] >= 3]  # Au moins 3 items

plt.figure(figsize=(12, 6))
category_stats['mean'].plot(kind='barh', color='teal')
plt.title('Calories Moyennes par Catégorie d\'Aliments (kcal/100g)', fontsize=14, fontweight='bold')
plt.xlabel('Calories Moyennes')
plt.ylabel('Catégorie')
plt.tight_layout()
plt.show()

## 6. Index Glycémique

In [ ]:
# Distribution de l'index glycémique
df_gi = df_nutrition[df_nutrition['glycemic_index'].notna()]

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(df_gi['glycemic_index'], bins=20, color='salmon', edgecolor='black')
plt.axvline(55, color='green', linestyle='--', label='IG Bas (<55)')
plt.axvline(70, color='orange', linestyle='--', label='IG Moyen (55-70)')
plt.title('Distribution de l\'Index Glycémique', fontweight='bold')
plt.xlabel('Index Glycémique')
plt.ylabel('Fréquence')
plt.legend()

# Box plot par catégorie
plt.subplot(1, 2, 2)
df_gi.boxplot(column='glycemic_index', by='category', ax=plt.gca(), rot=45)
plt.title('Index Glycémique par Catégorie', fontweight='bold')
plt.suptitle('')
plt.xlabel('Catégorie')
plt.ylabel('Index Glycémique')

plt.tight_layout()
plt.show()

## 7. Insights & Conclusions

### Observations clés :

1. **Distribution des macronutriments** : Identifier les aliments riches en protéines, glucides, ou lipides
2. **Corrélations** : Relations entre calories et macronutriments
3. **Index glycémique** : Proportion d'aliments adaptés aux diabétiques
4. **Catégories** : Profils nutritionnels par type d'aliment

### Applications pour le système de recommandation :

- Filtrage par contraintes nutritionnelles
- Scoring basé sur profil santé
- Recommandations d'alternatives équilibrées

In [ ]:
# Export des résultats
print("\n" + "="*60)
print("RÉSUMÉ DES INSIGHTS")
print("="*60)
print(f"Nombre total d'aliments analysés : {len(df_nutrition)}")
print(f"Nombre de plats répertoriés : {len(df_dishes)}")
print(f"\nCalories moyennes : {df_nutrition['energy_kcal'].mean():.1f} kcal/100g")
print(f"Protéines moyennes : {df_nutrition['protein_g'].mean():.1f} g/100g")
print(f"Glucides moyens : {df_nutrition['carbohydrate_g'].mean():.1f} g/100g")
print(f"Lipides moyens : {df_nutrition['fat_g'].mean():.1f} g/100g")
print(f"\nAliments avec IG bas (<55) : {len(df_gi[df_gi['glycemic_index'] < 55])} / {len(df_gi)}")
print(f"Aliments avec IG moyen (55-70) : {len(df_gi[(df_gi['glycemic_index'] >= 55) & (df_gi['glycemic_index'] < 70)])} / {len(df_gi)}")
print(f"Aliments avec IG élevé (>70) : {len(df_gi[df_gi['glycemic_index'] >= 70])} / {len(df_gi)}")
print("="*60)